In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
def load_and_prompt_test(trained=True, chat_template=True):
    model_id = "Qwen/Qwen2.5-7B"
    adapter_path = "./qwen-hermes-lora"

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    if trained:
        model = PeftModel.from_pretrained(model, adapter_path)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if chat_template:
        messages = [{"role": "user", "content": "Explain what a transformer neural network is in simple terms."}]
        
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
    else:
        inputs = tokenizer("Explain what a transformer neural network is", return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=300)
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(response)

In [ ]:
load_and_prompt_test(trained=True, chat_template=True)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
load_and_prompt_test(trained=False, chat_template=True)

In [ ]:
load_and_prompt_test(trained=False, chat_template=False)

In [ ]:
load_and_prompt_test(trained=True, chat_template=False)

In [1]:
!lm_eval --model hf \
    --model_args pretrained=Qwen/Qwen2.5-7B,peft=./qwen-hermes-lora,dtype=bfloat16 \
    --tasks mmlu \
    --num_fewshot 5 \
    --batch_size 2 \
    --output_path ./finetuned_results

/bin/bash: line 1: lm_eval: command not found
